# General Data Analysis

This notebook scans the contents of `data/`, loads each table, and runs a reusable exploratory analysis workflow.

It includes:
- dataset inventory and quick previews
- missingness, numeric, and categorical summaries for every table
- lightweight plotting helpers
- a focused section for `players_quarters_final.csv`, which is the main contest dataset


In [ ]:
from pathlib import Path
from typing import Dict

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
plt.style.use("ggplot")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


In [ ]:
def load_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    for col in df.select_dtypes(include="object").columns:
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        values = set(non_null.astype(str).str.strip().str.lower().unique())
        if values <= {"true", "false"}:
            df[col] = df[col].astype(str).str.strip().str.lower().map({"true": True, "false": False})
    return df


def build_inventory(data_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        if path.suffix.lower() not in {".csv", ".parquet", ".xlsx", ".xls"}:
            continue
        df = load_table(path)
        rows.append(
            {
                "file_name": path.name,
                "format": path.suffix.lower().lstrip("."),
                "rows": len(df),
                "columns": df.shape[1],
                "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
            }
        )
    return pd.DataFrame(rows).sort_values(["rows", "file_name"], ascending=[False, True]).reset_index(drop=True)


def dataset_overview(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })
    return out.sort_values(["missing_pct", "unique"], ascending=[False, False])


def numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=np.number)
    if num.empty:
        return pd.DataFrame()
    summary = num.describe().T
    summary["missing"] = df[num.columns].isna().sum()
    summary["missing_pct"] = (df[num.columns].isna().mean() * 100).round(2)
    return summary.sort_values("std", ascending=False)


def categorical_summary(df: pd.DataFrame, top_n: int = 8) -> Dict[str, pd.DataFrame]:
    result = {}
    cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
    for col in cat_cols:
        counts = df[col].fillna("<NA>").value_counts(dropna=False).head(top_n)
        result[col] = pd.DataFrame({
            "count": counts,
            "pct": (counts / len(df) * 100).round(2),
        })
    return result


def plot_top_missing(df: pd.DataFrame, title: str, top_n: int = 15) -> None:
    missing = (df.isna().mean() * 100).sort_values(ascending=False).head(top_n)
    if missing.max() == 0:
        print("No missing values in this table.")
        return
    ax = missing.sort_values().plot(kind="barh", figsize=(8, max(4, top_n * 0.35)), color="#C44E52")
    ax.set_title(f"Top Missingness: {title}")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("")
    plt.show()


def plot_numeric_distributions(df: pd.DataFrame, title: str, max_cols: int = 6) -> None:
    num_cols = df.select_dtypes(include=np.number).columns.tolist()[:max_cols]
    if not num_cols:
        print("No numeric columns to plot.")
        return
    ncols = 2
    nrows = math.ceil(len(num_cols) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, num_cols):
        df[col].dropna().plot(kind="hist", bins=30, ax=ax, color="#4C72B0", alpha=0.8)
        ax.set_title(col)
    for ax in axes[len(num_cols):]:
        ax.axis("off")
    fig.suptitle(f"Numeric Distributions: {title}", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
inventory = build_inventory(DATA_DIR)
tables = {
    path.name: load_table(path)
    for path in sorted(DATA_DIR.iterdir())
    if path.is_file() and path.suffix.lower() in {".csv", ".parquet", ".xlsx", ".xls"}
}

display(Markdown("## Data Inventory"))
display(inventory)

display(Markdown("## Quick Preview"))
for name, df in tables.items():
    display(Markdown(f"### `{name}`"))
    print(f"shape = {df.shape}")
    display(df.head())


In [ ]:
for name, df in tables.items():
    display(Markdown(f"## Overview: `{name}`"))
    display(dataset_overview(df))

    num_summary = numeric_summary(df)
    if not num_summary.empty:
        display(Markdown("### Numeric Summary"))
        display(num_summary)

    cat_summary = categorical_summary(df)
    if cat_summary:
        display(Markdown("### Top Categories"))
        for col, summary in cat_summary.items():
            display(Markdown(f"**{col}**"))
            display(summary)

    plot_top_missing(df, name)
    plot_numeric_distributions(df, name)


## Contest-Aware Analysis: `players_quarters_final.csv`

The documentation in `docs/` frames `players_quarters_final.csv` as the main modeling table:

- one row = one player snapshot at a 15-minute checkpoint within a match
- documented scope: `3,486` rows, `31` matches, `869` player appearances
- target `scored_after` is rare by construction, because goals after a checkpoint are rare events
- `last15_` and `cumul_` features come only from the run and shot source tables
- pass and behaviour-under-pressure tables were **not** used in the base panel and are available for extension
- shot outcome was intentionally removed from the contest version of the shot table to avoid leakage

The cells below turn those documentation details into explicit dataset checks and modeling-oriented EDA.


In [ ]:
main_df = tables["players_quarters_final.csv"].copy()
main_df["date"] = pd.to_datetime(main_df["date"])

checkpoint_order = ["H1_15", "H1_30", "H1_45", "H2_15", "H2_30", "H2_45", "ET1_15"]
checkpoint_to_abs_min = {
    "H1_15": 15,
    "H1_30": 30,
    "H1_45": 45,
    "H2_15": 60,
    "H2_30": 75,
    "H2_45": 90,
    "ET1_15": 105,
}
interval_index = {checkpoint: idx + 1 for idx, checkpoint in enumerate(checkpoint_order)}
feature_families = [
    ("sprints", "last15_sprints", "cumul_sprints"),
    ("hsr", "last15_hsr", "cumul_hsr"),
    ("distance", "last15_distance", "cumul_distance"),
    ("mean_max_speed", "last15_mean_max_speed", "cumul_mean_max_speed"),
    ("peak_speed", "last15_peak_speed", "cumul_peak_speed"),
    ("shots", "last15_shots", "cumul_shots"),
    ("shots_on_target", "last15_shots_on_target", "cumul_shots_on_target"),
    ("shots_under_press", "last15_shots_under_press", "cumul_shots_under_press"),
    ("shots_top_third", "last15_shots_top_third", "cumul_shots_top_third"),
]

main_df["checkpoint"] = pd.Categorical(main_df["checkpoint"], categories=checkpoint_order, ordered=True)
main_df["checkpoint_abs_min"] = main_df["checkpoint"].astype(str).map(checkpoint_to_abs_min)
main_df["interval_idx"] = main_df["checkpoint"].astype(str).map(interval_index)
main_df["active_at_checkpoint"] = (
    (main_df["minute_in"] <= main_df["checkpoint_abs_min"])
    & (main_df["minute_out"] >= main_df["checkpoint_abs_min"])
)
main_df["minutes_available_before_checkpoint"] = (
    main_df["checkpoint_abs_min"] - main_df["minute_in"] + 1
).clip(lower=1)
main_df["short_stint_before_checkpoint"] = main_df["minutes_available_before_checkpoint"] < 15

contest_summary = pd.DataFrame(
    [
        {"metric": "rows", "documented": "3,486", "observed": f"{len(main_df):,}"},
        {"metric": "unique matches", "documented": "31", "observed": str(main_df["fixture_id"].nunique())},
        {
            "metric": "unique player appearances",
            "documented": "869",
            "observed": str(main_df["player_appearance_id"].nunique()),
        },
        {
            "metric": "positive target rate",
            "documented": "203 / 3,486 (5.82%)",
            "observed": f"{main_df['scored_after'].sum():,} / {len(main_df):,} ({main_df['scored_after'].mean() * 100:.2f}%)",
        },
        {
            "metric": "rows active at checkpoint",
            "documented": "all rows should satisfy the checkpoint rule",
            "observed": f"{main_df['active_at_checkpoint'].sum():,} / {len(main_df):,}",
        },
        {
            "metric": "rows with <15 minutes available before checkpoint",
            "documented": "possible for recent substitutions",
            "observed": f"{main_df['short_stint_before_checkpoint'].sum():,} ({main_df['short_stint_before_checkpoint'].mean() * 100:.2f}%)",
        },
    ]
)

display(Markdown("### Documentation Consistency Checks"))
display(contest_summary)

checkpoint_profile = (
    main_df.groupby(["checkpoint", "checkpoint_period", "checkpoint_min"], observed=True)
    .agg(
        rows=("player_appearance_id", "size"),
        unique_appearances=("player_appearance_id", "nunique"),
        scored_after_rate=("scored_after", "mean"),
        subbed_share=("subbed", "mean"),
        short_stint_share=("short_stint_before_checkpoint", "mean"),
    )
    .reset_index()
)
checkpoint_profile["share_of_rows_pct"] = checkpoint_profile["rows"] / len(main_df) * 100
checkpoint_profile["scored_after_rate"] = (checkpoint_profile["scored_after_rate"] * 100).round(2)
checkpoint_profile["subbed_share"] = (checkpoint_profile["subbed_share"] * 100).round(2)
checkpoint_profile["short_stint_share"] = (checkpoint_profile["short_stint_share"] * 100).round(2)
checkpoint_profile["share_of_rows_pct"] = checkpoint_profile["share_of_rows_pct"].round(2)
display(Markdown("### Checkpoint Structure"))
display(checkpoint_profile)

context_summary = (
    main_df.groupby(["position", "is_home", "subbed"], observed=False)["scored_after"]
    .agg(["count", "mean"])
    .reset_index()
)
context_summary["mean"] = (context_summary["mean"] * 100).round(2)
context_summary = context_summary.rename(columns={"mean": "scored_after_rate_pct"})
display(Markdown("### Target Rate Across Position, Venue, and Substitution Status"))
display(context_summary.sort_values(["position", "subbed", "is_home"]))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

main_df.groupby("checkpoint", observed=False).size().plot(
    kind="bar",
    ax=axes[0, 0],
    color="#2E86AB",
)
axes[0, 0].set_title("Rows per Checkpoint")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("rows")

main_df.groupby("checkpoint", observed=False)["scored_after"].mean().mul(100).plot(
    kind="bar",
    ax=axes[0, 1],
    color="#4CAF50",
)
axes[0, 1].set_title("Scored-After Rate by Checkpoint (%)")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("rate (%)")

main_df.groupby("position", observed=False)["scored_after"].mean().mul(100).sort_values(ascending=False).plot(
    kind="bar",
    ax=axes[0, 2],
    color="#C44E52",
)
axes[0, 2].set_title("Scored-After Rate by Position (%)")
axes[0, 2].set_xlabel("")
axes[0, 2].set_ylabel("rate (%)")

main_df.groupby("checkpoint_period", observed=False).size().plot(
    kind="bar",
    ax=axes[1, 0],
    color="#8172B2",
)
axes[1, 0].set_title("Rows by Match Period")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("rows")

context_plot = pd.DataFrame(
    {
        "home": main_df.groupby("is_home")["scored_after"].mean().mul(100),
        "subbed": main_df.groupby("subbed")["scored_after"].mean().mul(100),
    }
)
context_plot.index = [str(index) for index in context_plot.index]
context_plot.plot(kind="bar", ax=axes[1, 1], color=["#DD8452", "#55A868"])
axes[1, 1].set_title("Home/Away and Substitution Context (%)")
axes[1, 1].set_xlabel("flag value")
axes[1, 1].set_ylabel("rate (%)")
axes[1, 1].legend(loc="best")

main_df["formation"].value_counts().head(10).sort_values().plot(
    kind="barh",
    ax=axes[1, 2],
    color="#937860",
)
axes[1, 2].set_title("Most Common Formations")
axes[1, 2].set_xlabel("rows")
axes[1, 2].set_ylabel("")

plt.tight_layout()
plt.show()


## Recent vs. Cumulative Signals

The problem statement explicitly asks whether recent 15-minute behavior or cumulative behavior is more informative, and whether short-term intensity relative to the player's overall level matters.

The next cell turns that into two simple checks:

- compare `last15_*` correlations with their `cumul_*` counterparts
- compute the recent share of each metric (`last15 / cumul`) to see whether late concentration of actions is associated with a higher `scored_after` rate


In [ ]:
feature_strength_rows = []
recent_share_rows = []

for family, recent_col, cumul_col in feature_families:
    last15_corr = main_df[recent_col].corr(main_df["scored_after"])
    cumul_corr = main_df[cumul_col].corr(main_df["scored_after"])
    feature_strength_rows.append(
        {
            "feature_family": family,
            "last15_corr": last15_corr,
            "cumul_corr": cumul_corr,
            "stronger_signal": "last15" if abs(last15_corr) > abs(cumul_corr) else "cumul",
            "abs_corr_gap": abs(last15_corr) - abs(cumul_corr),
        }
    )

    recent_share = main_df[recent_col] / main_df[cumul_col].replace(0, np.nan)
    subset = main_df.loc[recent_share.notna(), ["scored_after"]].copy()
    subset["high_recent_share"] = recent_share[recent_share.notna()] >= 0.5
    if not subset.empty:
        grouped = subset.groupby("high_recent_share")["scored_after"].agg(["count", "mean"]).reset_index()
        grouped["feature_family"] = family
        grouped["mean"] = (grouped["mean"] * 100).round(2)
        recent_share_rows.append(grouped)

feature_strength = pd.DataFrame(feature_strength_rows).sort_values("abs_corr_gap", ascending=False)
feature_strength[["last15_corr", "cumul_corr", "abs_corr_gap"]] = feature_strength[
    ["last15_corr", "cumul_corr", "abs_corr_gap"]
].round(4)

display(Markdown("### `last15_` vs `cumul_` Correlation Comparison"))
display(feature_strength)

recent_share_effect = pd.concat(recent_share_rows, ignore_index=True)
recent_share_effect = recent_share_effect.rename(
    columns={
        "count": "rows",
        "mean": "scored_after_rate_pct",
        "high_recent_share": "recent_share_ge_50pct",
    }
)
display(Markdown("### Does Recent Share Matter?"))
display(recent_share_effect)

plot_families = ["shots", "sprints", "hsr", "distance"]
plot_df = recent_share_effect[recent_share_effect["feature_family"].isin(plot_families)].copy()
plot_df["recent_share_ge_50pct"] = plot_df["recent_share_ge_50pct"].map({False: "< 50% of cumulative", True: ">= 50% of cumulative"})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
feature_strength.set_index("feature_family")[["last15_corr", "cumul_corr"]].plot(
    kind="bar",
    ax=axes[0],
    color=["#4C72B0", "#C44E52"],
)
axes[0].set_title("Recent vs. Cumulative Correlations")
axes[0].set_xlabel("")
axes[0].set_ylabel("correlation with scored_after")
axes[0].tick_params(axis="x", rotation=45)

for feature_family, group in plot_df.groupby("feature_family"):
    axes[1].plot(
        group["recent_share_ge_50pct"],
        group["scored_after_rate_pct"],
        marker="o",
        linewidth=2,
        label=feature_family,
    )
axes[1].set_title("Higher Recent Share Often Means Higher Target Rate")
axes[1].set_xlabel("")
axes[1].set_ylabel("scored_after rate (%)")
axes[1].legend(loc="best")

plt.tight_layout()
plt.show()


## Supplementary Tables and Feature Space Extensions

The docs make an important distinction:

- `player_appearance_run.csv` and `player_appearance_shot_limited.csv` already feed the base panel
- `player_appearance_pass.csv` and `player_appearance_behaviour_under_pressure.csv` are **unused** in the base table and are the cleanest source of additional explanatory power
- event tables may contain player appearances that never show up in the checkpoint panel, because the main table includes only players active at defined checkpoints

That means any future join should be built at the **checkpoint level** using event `period` and `minute`, not by attaching full-match aggregates to every checkpoint row.


In [ ]:
supplementary_docs = {
    "player_appearance_pass.csv": 29795,
    "player_appearance_behaviour_under_pressure.csv": 12185,
    "player_appearance_run.csv": 35133,
    "player_appearance_shot_limited.csv": 780,
}
main_appearance_ids = set(main_df["player_appearance_id"].unique())

supplementary_summary = []
for name, documented_rows in supplementary_docs.items():
    event_df = tables[name]
    event_appearance_ids = set(event_df["player_appearance_id"].dropna().astype(int).unique())
    overlap = len(main_appearance_ids & event_appearance_ids)
    supplementary_summary.append(
        {
            "table": name,
            "documented_rows": documented_rows,
            "observed_rows": len(event_df),
            "event_unique_appearances": len(event_appearance_ids),
            "overlap_with_main_panel": overlap,
            "event_only_appearances": len(event_appearance_ids - main_appearance_ids),
            "main_panel_without_this_event_table": len(main_appearance_ids - event_appearance_ids),
        }
    )

display(Markdown("### Supplementary Table Coverage vs. Main Panel"))
display(pd.DataFrame(supplementary_summary).sort_values("observed_rows", ascending=False))

appearance_base = main_df.groupby("player_appearance_id", observed=False).agg(
    position=("position", "first"),
    checkpoints=("checkpoint", "size"),
    scored_after_any_checkpoint=("scored_after", "max"),
)

passes = tables["player_appearance_pass.csv"].copy()
pass_features = passes.groupby("player_appearance_id").agg(
    passes=("id", "count"),
    pass_accuracy=("accurate", "mean"),
    pass_top_share=("stage", lambda s: (s == "top").mean()),
    no_target_share=("addressee_player_appearance_id", lambda s: s.isna().mean()),
)

pressure = tables["player_appearance_behaviour_under_pressure.csv"].copy()
pressure_features = pressure.groupby("player_appearance_id").agg(
    pressured_events=("id", "count"),
    pressured_accuracy=("accurate", "mean"),
    turnover_rate=("press_induced_outcome", lambda s: (s == "turnover").mean()),
    forward_under_pressure_rate=("press_induced_outcome", lambda s: (s == "forward_pass").mean()),
    mean_pass_angle=("pass_angle", "mean"),
)

runs = tables["player_appearance_run.csv"].copy()
run_features = runs.groupby("player_appearance_id").agg(
    run_events=("id", "count"),
    sprint_share=("run_type", lambda s: (s == "sprint").mean()),
    top_stage_run_share=("stage", lambda s: (s == "top").mean()),
    avg_run_distance=("distance", "mean"),
    avg_max_speed=("max_speed", "mean"),
)

shots = tables["player_appearance_shot_limited.csv"].copy()
shot_features = shots.groupby("player_appearance_id").agg(
    shot_events=("id", "count"),
    shot_under_pressure_rate=("under_pressure", "mean"),
    top_stage_shot_rate=("stage", lambda s: (s == "top").mean()),
    header_rate=("body_part", lambda s: (s == "head").mean()),
    set_piece_rate=(
        "play_pattern",
        lambda s: s.isin(["corner_kick", "direct_free_kick", "indirect_free_kick", "penalty", "throw_in"]).mean(),
    ),
)

candidate_blocks = {
    "pass-derived": pass_features,
    "pressure-derived": pressure_features,
    "run-context-derived": run_features,
    "shot-context-derived": shot_features,
}

feature_block_rows = []
for block_name, feature_df in candidate_blocks.items():
    merged = appearance_base.join(feature_df)
    available = merged[feature_df.columns].notna().any(axis=1)
    feature_block_rows.append(
        {
            "feature_block": block_name,
            "coverage_in_main_appearances_pct": round(available.mean() * 100, 2),
            "target_rate_when_available_pct": round(merged.loc[available, "scored_after_any_checkpoint"].mean() * 100, 2),
            "features_in_block": feature_df.shape[1],
        }
    )

display(Markdown("### Candidate Feature Blocks Available for Enrichment"))
display(pd.DataFrame(feature_block_rows).sort_values("coverage_in_main_appearances_pct", ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

passes.groupby("stage")["accurate"].mean().mul(100).reindex(["bottom", "middle", "top"]).plot(
    kind="bar",
    ax=axes[0, 0],
    color="#4C72B0",
)
axes[0, 0].set_title("Pass Accuracy by Pitch Zone (%)")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("accuracy (%)")

pressure.assign(is_turnover=pressure["press_induced_outcome"].eq("turnover")).groupby("stage")["is_turnover"].mean().mul(100).reindex(["bottom", "middle", "top"]).plot(
    kind="bar",
    ax=axes[0, 1],
    color="#C44E52",
)
axes[0, 1].set_title("Turnovers Under Pressure by Zone (%)")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("turnover rate (%)")

runs["run_type"].value_counts().reindex(["hsr", "sprint"]).plot(
    kind="bar",
    ax=axes[1, 0],
    color="#55A868",
)
axes[1, 0].set_title("Run Event Mix")
axes[1, 0].set_xlabel("")
axes[1, 0].set_ylabel("events")

shots["play_pattern"].value_counts().head(7).sort_values().plot(
    kind="barh",
    ax=axes[1, 1],
    color="#8172B2",
)
axes[1, 1].set_title("Shot Events by Play Pattern")
axes[1, 1].set_xlabel("events")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plt.show()


## Suggested Next Steps

- Build **checkpoint-aligned** pass and pressure aggregates using event `period` and `minute`, rather than joining full-match totals to every checkpoint row.
- Normalize activity by opportunity: position, minutes available before checkpoint, and whether the player was recently subbed in all clearly matter.
- Keep a strict leakage boundary: the docs explicitly remove shot outcome from the contest table, so any engineered shot features should use only safe context (`body_part`, `technique`, `play_pattern`, `stage`, `under_pressure`).
- Treat the panel structure seriously during modeling. Multiple rows come from the same player appearance and match, so validation should split by `fixture_id` or at least `player_appearance_id`, not by random row.
